In [1]:
import os
import numpy as np
import librosa
import xml.etree.ElementTree as ET
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix


# Function to parse annotations from RML files
def parse_annotations(rml_file_path):
    annotations = []
    try:
        tree = ET.parse(rml_file_path)
        root = tree.getroot()
        for event in root.findall(".//ns0:Event", namespaces={"ns0": "http://www.respironics.com/PatientStudy.xsd"}):
            family = event.attrib.get("Family")
            type_ = event.attrib.get("Type")
            start = float(event.attrib.get("Start"))
            duration = float(event.attrib.get("Duration"))
            end = start + duration
            if family == "Respiratory":  # Only focus on respiratory annotations
                annotations.append({"family": family, "type": type_, "start": start, "end": end})
    except Exception as e:
        print(f"Error parsing {rml_file_path}: {e}")
    return annotations


# Function to extract statistical features from audio
def extract_statistical_features(wav_file_path, annotations, sr=22050):
    features = []
    labels = []

    try:
        audio, _ = librosa.load(wav_file_path, sr=sr)
        for annotation in annotations:
            start_sample = int(annotation["start"] * sr)
            end_sample = int(annotation["end"] * sr)

            if end_sample > len(audio):
                continue  # Skip annotations outside the range of the audio file

            segment = audio[start_sample:end_sample]

            # Extract statistical features
            zcr = np.mean(librosa.feature.zero_crossing_rate(y=segment))
            rmse = np.mean(librosa.feature.rms(y=segment))
            spectral_centroid = np.mean(librosa.feature.spectral_centroid(y=segment, sr=sr))
            spectral_bandwidth = np.mean(librosa.feature.spectral_bandwidth(y=segment, sr=sr))
            spectral_flatness = np.mean(librosa.feature.spectral_flatness(y=segment))
            spectral_rolloff = np.mean(librosa.feature.spectral_rolloff(y=segment, sr=sr))

            # Combine features
            feature_vector = [zcr, rmse, spectral_centroid, spectral_bandwidth, spectral_flatness, spectral_rolloff]
            features.append(feature_vector)
            labels.append(annotation["type"])
    except Exception as e:
        print(f"Error processing file {wav_file_path}: {e}")

    return np.array(features), labels


# Main function to process folders and train Random Forest Classifier
def process_and_train_random_forest(wav_folder_path, rml_folder_path, sr=22050):
    all_features = []
    all_labels = []

    wav_files = sorted([f for f in os.listdir(wav_folder_path) if f.endswith(".wav")])
    rml_files = sorted([f for f in os.listdir(rml_folder_path) if f.endswith(".rml")])

    for wav_file, rml_file in zip(wav_files, rml_files):
        wav_file_path = os.path.join(wav_folder_path, wav_file)
        rml_file_path = os.path.join(rml_folder_path, rml_file)

        annotations = parse_annotations(rml_file_path)
        features, labels = extract_statistical_features(wav_file_path, annotations, sr)
        if len(features) > 0:
            all_features.extend(features)
            all_labels.extend(labels)

    if not all_features or not all_labels:
        print("No data available for training. Check the file paths and ensure annotations exist in the RML files.")
        return

    # Encode labels
    label_encoder = LabelEncoder()
    all_labels_encoded = label_encoder.fit_transform(all_labels)

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(all_features, all_labels_encoded, test_size=0.2, random_state=42)

    # Train Random Forest Classifier
    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    clf.fit(X_train, y_train)

    # Evaluate the model
    y_pred = clf.predict(X_test)
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    # Compute and print ROC AUC scores
    y_test_binarized = label_binarize(y_test, classes=range(len(label_encoder.classes_)))
    roc_auc_scores = {}
    for i, class_name in enumerate(label_encoder.classes_):
        roc_auc = roc_auc_score(y_test_binarized[:, i], y_pred_proba[:, i])
        roc_auc_scores[class_name] = roc_auc

    print("\nROC AUC Scores:")
    for class_name, roc_auc in roc_auc_scores.items():
        print(f"Class {class_name}: {roc_auc:.4f}")



# Specify the folder paths
wav_folder_path = "/Users/terlan/Library/CloudStorage/OneDrive-uni-mannheim.de/cleaned_wav/Mic"
rml_folder_path = "/Users/terlan/Library/CloudStorage/OneDrive-uni-mannheim.de/cleaned_label"

# Execute
process_and_train_random_forest(wav_folder_path, rml_folder_path)



Classification Report:
                  precision    recall  f1-score   support

    CentralApnea       0.29      0.09      0.14        90
        Hypopnea       0.61      0.47      0.53       593
      MixedApnea       0.44      0.28      0.34       369
ObstructiveApnea       0.65      0.81      0.72      1329

        accuracy                           0.62      2381
       macro avg       0.50      0.41      0.43      2381
    weighted avg       0.59      0.62      0.59      2381


Confusion Matrix:
[[   8   13   17   52]
 [   2  276   24  291]
 [  13   18  104  234]
 [   5  149   92 1083]]


In [ ]:
from sklearn.metrics import roc_auc_score

print(roc_auc_score())